# Exp 12 (Mini Project): Stock price / returns prediction - Indian market 2000-2020
**Dataset:** https://www.kaggle.com/datasets/sagara9595/stock-data  (download the CSV and set `FILE` below to its name)

Run the first code cell and look at the printed column names; the code picks the *Date* and *Close* columns automatically.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

FILE = "stock_data.csv"        # <- change to the name of the downloaded CSV
df = pd.read_csv(FILE)
print(df.columns.tolist())
print(df.head())
print(df.info())

In [ ]:
# find the Date and Close columns whatever their capitalisation
date_col = [c for c in df.columns if "date" in c.lower()][0]
close_col = [c for c in df.columns if c.lower() in ("close", "close price", "adj close")][0]

df[date_col] = pd.to_datetime(df[date_col])
df = df.sort_values(date_col).set_index(date_col)
df = df.dropna(subset=[close_col])
print("Using:", date_col, "and", close_col)
print(df[close_col].describe())

## Analyse the ups and downs of the market

In [ ]:
df["Return"] = df[close_col].pct_change()
df["MA50"] = df[close_col].rolling(50).mean()
df["MA200"] = df[close_col].rolling(200).mean()

plt.figure(figsize=(12, 5))
plt.plot(df[close_col], label="Close")
plt.plot(df["MA50"], label="50-day MA")
plt.plot(df["MA200"], label="200-day MA")
plt.title("Closing price with moving averages"); plt.legend(); plt.show()

plt.figure(figsize=(12, 3))
plt.plot(df["Return"], linewidth=0.6)
plt.title("Daily returns (ups and downs)"); plt.show()

print("Days market went up  :", (df["Return"] > 0).sum())
print("Days market went down:", (df["Return"] < 0).sum())

## Predict future price returns

In [ ]:
# Features: previous 5 days' returns, target: next day's return
data = pd.DataFrame({"Return": df["Return"]})
for lag in range(1, 6):
    data[f"lag{lag}"] = data["Return"].shift(lag)
data["MA_ratio"] = (df[close_col] / df["MA50"]) - 1
data = data.dropna()

X = data.drop(columns=["Return"])
y = data["Return"]
split = int(len(data) * 0.8)                    # time-ordered split
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

lr = LinearRegression().fit(X_train, y_train)
rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1).fit(X_train, y_train)
pred_lr = lr.predict(X_test)
pred_rf = rf.predict(X_test)

In [ ]:
def ev(name, pred):
    return {"Model": name, "R2": r2_score(y_test, pred),
            "MAE": mean_absolute_error(y_test, pred),
            "RMSE": np.sqrt(mean_squared_error(y_test, pred))}
print(pd.DataFrame([ev("Linear Regression", pred_lr), ev("Random Forest", pred_rf)]))

plt.figure(figsize=(12, 4))
plt.plot(y_test.index, y_test.values, label="Actual", linewidth=0.7)
plt.plot(y_test.index, pred_rf, label="Predicted (RF)", linewidth=0.7)
plt.title("Actual vs predicted returns"); plt.legend(); plt.show()

**Conclusion:** The ups and downs of the market were analysed and future returns predicted. Stock returns are very noisy, so R2 close to 0 is normal.